# Quantinuum H2 Real-Hardware [[3,1,1]] QEC Demonstration

This notebook runs a minimal **3-qubit repetition-code QEC feedback experiment** on real Quantinuum H2 hardware.

Workflow:

$$
|000\rangle \xrightarrow{X_1} |010\rangle
\rightarrow \text{measure } Z_0Z_1,\ Z_1Z_2
\rightarrow s=(1,1)
\rightarrow \text{real-time conditional }X_1
\rightarrow |000\rangle .
$$

The decoder in this notebook is a **hard-coded classical lookup rule inside the circuit**, not a Wasm decoder.

**Run All** is intended to work from a fresh Nexus kernel with no manually recovered variables.  
The notebook explicitly creates/sets the project, compiles for the selected H2 target, runs the matching syntax checker, checks the estimated HQC cost against a safety cap, submits the real-hardware job, waits for completion, and downloads the results.

> **Important:** with `SUBMIT_HARDWARE = True`, running this notebook submits a real hardware job and consumes HQC. Re-running it submits another job.

In [1]:
# ============================================================
# 0. IMPORTS AND USER SETTINGS
# ============================================================

import importlib.metadata as md
import uuid
import time

import qnexus as qnx

from pytket.circuit import Bit, Circuit
from pytket.circuit.logic_exp import reg_eq


# -----------------------------
# User settings
# -----------------------------

PROJECT_NAME = "Real Hardware QEC Benchmark"

# Change to "H2-2" if you intentionally want the other H2 machine.
TARGET = "H2-1"

N_SHOTS = 20

# Hard safety cap for the real-hardware submission.
# The syntax checker reports the predicted HQC cost first.
MAX_COST_HQC = 6.0

# Set False if you only want compile + syntax check + cost estimate.
SUBMIT_HARDWARE = True


# -----------------------------
# Derived settings
# -----------------------------

SYNTAX_CHECKER = f"{TARGET}SC"
RUN_ID = str(uuid.uuid4())[:8]

project = qnx.projects.get_or_create(name=PROJECT_NAME)
qnx.context.set_active_project(project)


def _status_value(job_status):
    """Return a plain status string such as COMPLETED or ERROR."""
    value = getattr(job_status.status, "value", None)
    return value if value is not None else str(job_status.status)


def _require_completed(label, job_status):
    """Fail clearly if a Nexus job did not complete successfully."""
    value = _status_value(job_status)
    if value != "COMPLETED":
        raise RuntimeError(
            f"{label} did not complete successfully.\n"
            f"Status: {job_status}"
        )


def _wait_with_status(job, label, poll_seconds=10):
    """Poll a Nexus job and print status changes while waiting."""
    last_value = None

    while True:
        status = qnx.jobs.status(job)
        value = _status_value(status)

        if value != last_value:
            print(f"{label}: {value}", flush=True)
            last_value = value

        if value in {"COMPLETED", "ERROR", "CANCELLED"}:
            return status

        time.sleep(poll_seconds)


print("qnexus version :", md.version("qnexus"))
print("pytket version :", md.version("pytket"))
print("Project        :", PROJECT_NAME)
print("Target         :", TARGET)
print("Syntax checker :", SYNTAX_CHECKER)
print("Shots          :", N_SHOTS)
print("Max HQC        :", MAX_COST_HQC)
print("Run ID         :", RUN_ID)

qnexus version : 0.49.0
pytket version : 2.18.1
Project        : Real Hardware QEC Benchmark
Target         : H2-1
Syntax checker : H2-1SC
Shots          : 20
Max HQC        : 6.0
Run ID         : bd7ab413


In [2]:
# ============================================================
# 1. BUILD THE 3-QUBIT REPETITION-CODE QEC CIRCUIT
# ============================================================

circuit = Circuit()
circuit.name = "3-qubit-repetition-real-hardware"

# Quantum registers
data = circuit.add_q_register("data", 3)
ancilla = circuit.add_q_register("ancilla", 1)

# Classical registers
syndrome = circuit.add_c_register("syndrome", 2)
output = circuit.add_c_register("output", 3)


# ------------------------------------------------------------
# Logical state
# ------------------------------------------------------------
# Qubits start in |0>, so the data register begins in:
#
#     |0_L> = |000>
#
# Deliberately inject one X error on the middle data qubit:
#
#     |000> -> |010>
# ------------------------------------------------------------

circuit.X(data[1])


# ------------------------------------------------------------
# Syndrome bit s0 from Z0 Z1
#
# For computational-basis states:
#     s0 = d0 XOR d1
# ------------------------------------------------------------

circuit.CX(data[0], ancilla[0])
circuit.CX(data[1], ancilla[0])
circuit.Measure(ancilla[0], syndrome[0])
circuit.Reset(ancilla[0])


# ------------------------------------------------------------
# Syndrome bit s1 from Z1 Z2
#
#     s1 = d1 XOR d2
# ------------------------------------------------------------

circuit.CX(data[1], ancilla[0])
circuit.CX(data[2], ancilla[0])
circuit.Measure(ancilla[0], syndrome[1])
circuit.Reset(ancilla[0])


# ------------------------------------------------------------
# Real-time classical decoder + quantum feed-forward
#
# pytket interprets the 2-bit register as:
#     integer = s0 + 2*s1
#
# (s0,s1) = (1,0) -> 1 -> X on data[0]
# (s0,s1) = (1,1) -> 3 -> X on data[1]
# (s0,s1) = (0,1) -> 2 -> X on data[2]
# ------------------------------------------------------------

circuit.X(data[0], condition=reg_eq(syndrome, 1))
circuit.X(data[1], condition=reg_eq(syndrome, 3))
circuit.X(data[2], condition=reg_eq(syndrome, 2))


# ------------------------------------------------------------
# Final data measurement
# ------------------------------------------------------------

for i in range(3):
    circuit.Measure(data[i], output[i])


print("Circuit created")
print(circuit)
print(f"Qubits: {circuit.n_qubits}")
print(f"Bits:   {circuit.n_bits}")

Circuit created
<tket::Circuit, qubits=4, gates=18>
Qubits: 4
Bits:   8


In [ ]:
# ============================================================
# 2. UPLOAD -> COMPILE -> SYNTAX CHECK -> REAL HARDWARE
# ============================================================

# ------------------------------------------------------------
# Upload
# ------------------------------------------------------------

circuit_ref = qnx.circuits.upload(
    circuit=circuit,
    name=f"real-qec-{TARGET}-{RUN_ID}",
    project=project,
)

print("\n[1/4] Circuit uploaded")
print("CircuitRef ID:", circuit_ref.id)


# ------------------------------------------------------------
# Compile specifically for the real H2 target
# ------------------------------------------------------------

hardware_config = qnx.QuantinuumConfig(
    device_name=TARGET
)

compile_job = qnx.start_compile_job(
    programs=[circuit_ref],
    backend_config=hardware_config,
    optimisation_level=0,
    project=project,
    name=f"compile-real-qec-{TARGET}-{RUN_ID}",
)

print("\n[2/4] Compilation submitted")
print("Compile job ID:", compile_job.id)

compile_status = _wait_with_status(
    compile_job,
    "Compilation"
)
print("Compilation status:", compile_status)

_require_completed("Compilation", compile_status)

compiled_ref = qnx.jobs.results(compile_job)[0].get_output()

print("Compiled CircuitRef ID:", compiled_ref.id)


# ------------------------------------------------------------
# Syntax checker
# ------------------------------------------------------------

sc_config = qnx.QuantinuumConfig(
    device_name=SYNTAX_CHECKER
)

sc_job = qnx.start_execute_job(
    programs=[compiled_ref],
    n_shots=[N_SHOTS],
    backend_config=sc_config,
    project=project,
    name=f"syntax-check-real-qec-{TARGET}-{RUN_ID}",
)

print("\n[3/4] Syntax checker submitted")
print("Syntax-check job ID:", sc_job.id)

sc_status = _wait_with_status(
    sc_job,
    "Syntax check"
)
print("Syntax-check status:", sc_status)

_require_completed("Syntax check", sc_status)

estimated_cost = sc_status.cost

print("Estimated hardware cost:", estimated_cost, "HQC")


# ------------------------------------------------------------
# Cost guard
# ------------------------------------------------------------

if estimated_cost is not None and estimated_cost > MAX_COST_HQC:
    raise RuntimeError(
        f"Predicted cost {estimated_cost} HQC exceeds "
        f"MAX_COST_HQC={MAX_COST_HQC}. "
        "Hardware job was NOT submitted."
    )

if not SUBMIT_HARDWARE:
    print(
        "\nSUBMIT_HARDWARE=False: stopping after successful "
        "syntax check. No real-hardware job was submitted."
    )
else:
    # --------------------------------------------------------
    # Real H2 hardware submission
    # --------------------------------------------------------

    hardware_job = qnx.start_execute_job(
        programs=[compiled_ref],
        n_shots=[N_SHOTS],
        backend_config=hardware_config,
        project=project,
        max_cost=MAX_COST_HQC,
        name=f"REAL-QEC-{TARGET}-{RUN_ID}",
    )

    print("\n[4/4] REAL HARDWARE JOB SUBMITTED")
    print("Hardware job ID:", hardware_job.id)
    print("Initial status:", qnx.jobs.status(hardware_job))

    print("\nWaiting for real hardware...")
    final_status = _wait_with_status(
        hardware_job,
        "Real hardware"
    )
    print("Final hardware status:", final_status)

    _require_completed("Real-hardware execution", final_status)

    print("Actual HQC cost:", final_status.cost)


[1/4] Circuit uploaded
CircuitRef ID: 47b7cb4d-5a81-49b5-be38-916c36d73ba8

[2/4] Compilation submitted
Compile job ID: 4915d57b-0fcb-4565-8e68-bcc5953c424f
Compilation status: JobStatus(status=<JobStatusEnum.COMPLETED: 'COMPLETED'>, message='The job is completed.', error_detail=None, completed_time=datetime.datetime(2026, 9, 24, 12, 15, 46, 835441, tzinfo=datetime.timezone.utc), queued_time=datetime.datetime(2026, 9, 24, 12, 15, 42, 646185, tzinfo=datetime.timezone.utc), submitted_time=datetime.datetime(2026, 9, 24, 12, 15, 40, 888184, tzinfo=datetime.timezone.utc), running_time=datetime.datetime(2026, 9, 24, 12, 15, 43, 287137, tzinfo=datetime.timezone.utc), cancelled_time=None, error_time=None, queue_position=None, cost=None)
Compiled CircuitRef ID: 01af01d7-edbb-45ba-92cc-59e4697c1000

[3/4] Syntax checker submitted
Syntax-check job ID: f76e50c0-d660-4374-a534-85ec7cbabc5d


In [8]:
# ============================================================
# 3. DOWNLOAD AND SUMMARIZE RESULTS
# ============================================================

if not SUBMIT_HARDWARE:
    print("No hardware result to download because SUBMIT_HARDWARE=False.")
else:
    result = qnx.jobs.results(hardware_job)[0].download_result()

    # Reconstruct classical bits by register name.
    # This remains robust even if notebook variables are later recreated.
    s0 = Bit("syndrome", 0)
    s1 = Bit("syndrome", 1)

    o0 = Bit("output", 0)
    o1 = Bit("output", 1)
    o2 = Bit("output", 2)

    syndrome_counts = result.get_counts([s0, s1])
    output_counts = result.get_counts([o0, o1, o2])

    expected_syndrome = (1, 1)
    expected_output = (0, 0, 0)

    syndrome_matches = syndrome_counts.get(expected_syndrome, 0)
    strict_successes = output_counts.get(expected_output, 0)

    # Logical |0_L> is preserved by final majority vote if at most one bit is 1.
    logical_successes = sum(
        count
        for bits, count in output_counts.items()
        if sum(bits) <= 1
    )

    syndrome_match_rate = syndrome_matches / N_SHOTS
    strict_success_rate = strict_successes / N_SHOTS
    logical_success_rate = logical_successes / N_SHOTS

    print("\n" + "=" * 60)
    print("REAL H2 QEC DEMONSTRATION RESULT")
    print("=" * 60)

    print("\nSyndrome counts")
    print(syndrome_counts)

    print("\nFinal data counts")
    print(output_counts)

    print("\nSummary")
    print(f"Target                    : {TARGET}")
    print(f"Shots                     : {N_SHOTS}")
    print(f"Expected syndrome         : {expected_syndrome}")
    print(f"Syndrome match rate       : {syndrome_match_rate:.6f}")
    print(f"Expected final state      : {expected_output}")
    print(f"Strict |000> success rate : {strict_success_rate:.6f}")
    print(f"Logical-0 success rate    : {logical_success_rate:.6f}")
    print(f"Estimated HQC cost        : {estimated_cost}")
    print(f"Actual HQC cost           : {final_status.cost}")

    print("\nResource IDs")
    print("CircuitRef     :", circuit_ref.id)
    print("CompiledRef    :", compiled_ref.id)
    print("CompileJob     :", compile_job.id)
    print("SyntaxCheckJob :", sc_job.id)
    print("HardwareJob    :", hardware_job.id)

    print("\nFinished.")

Unknown OpType in BackendInfo: `RZZ`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `Rxxyyzz`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `U1q`, will omit from BackendInfo. Consider updating your pytket version.
Unknown OpType in BackendInfo: `ZZ`, will omit from BackendInfo. Consider updating your pytket version.



REAL H2 QEC BENCHMARK RESULT

Syndrome counts
Counter({(1, 1): 20})

Final data counts
Counter({(0, 0, 0): 20})

Summary
Target                    : H2-1
Shots                     : 20
Expected syndrome         : (1, 1)
Syndrome match rate       : 1.000000
Expected final state      : (0, 0, 0)
Strict |000> success rate : 1.000000
Logical-0 success rate    : 1.000000
Estimated HQC cost        : 5.42
Actual HQC cost           : 5.42

Resource IDs
CircuitRef     : 47b7cb4d-5a81-49b5-be38-916c36d73ba8
CompiledRef    : 01af01d7-edbb-45ba-92cc-59e4697c1000
CompileJob     : 4915d57b-0fcb-4565-8e68-bcc5953c424f
SyntaxCheckJob : f76e50c0-d660-4374-a534-85ec7cbabc5d
HardwareJob    : 8b008f83-c0db-4c52-87aa-3324c392c67f

Finished.


## Interpretation

For the deliberately injected middle-qubit error $X_1$:

- expected syndrome: $(s_0,s_1)=(1,1)$;
- the in-circuit classical condition selects $X_1$;
- ideal final physical state: $|000\rangle$.

`Strict |000> success rate` requires all three final physical bits to be zero.  
`Logical-0 success rate` is more permissive: it applies a final repetition-code majority vote, so `000`, `001`, `010`, and `100` all decode to logical $0$.

The experiment is a minimal real-time QEC feedback demonstration. It does **not** use a Wasm decoder; the lookup-table decoder is embedded directly in the circuit through classical conditions.